# Clase 4 — Pandas I: lectura y exploración de datos

**Python y Políticas Públicas**

---

## Contenidos
1. ¿Qué es pandas?
2. Series y DataFrames
3. Crear DataFrames desde cero
4. Leer archivos: CSV, Excel, y otros formatos
5. Exploración inicial de un dataset
6. Selección básica de datos
7. Ejercicios

In [ ]:
import pandas as pd
import numpy as np

# Mostrar todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

---
## 1. ¿Qué es pandas?

`pandas` es la librería central para el análisis de datos en Python. Permite:

- Leer y escribir datos en docenas de formatos (CSV, Excel, JSON, SQL, Parquet, etc.)
- Manipular tablas de datos de forma intuitiva (similar a Excel, pero programático y escalable)
- Combinar, limpiar, transformar y resumir datos
- Integrarse perfectamente con matplotlib para visualización

Si venís de Stata o R, pandas es el equivalente a los DataFrames de R o las bases de datos de Stata.

---
## 2. Series y DataFrames

pandas tiene dos estructuras de datos fundamentales:
- **`Series`**: una columna de datos (array 1D con etiquetas)
- **`DataFrame`**: una tabla (colección de Series)

In [ ]:
# Series
pobreza = pd.Series(
    [42.3, 38.1, 39.7, 34.2, 48.6],
    index=["Buenos Aires", "Córdoba", "Santa Fe", "Mendoza", "Tucumán"],
    name="tasa_pobreza_2023"
)

print(pobreza)
print(f"\nMedia: {pobreza.mean():.1f}%")
print(f"Provincia con mayor pobreza: {pobreza.idxmax()}")

In [ ]:
# DataFrame: es la estructura que más usaremos
df = pd.DataFrame({
    "provincia": ["Buenos Aires", "Córdoba", "Santa Fe", "Mendoza", "Tucumán"],
    "poblacion": [17_500_000, 3_900_000, 3_600_000, 1_950_000, 1_700_000],
    "pobreza_pct": [42.3, 38.1, 39.7, 34.2, 48.6],
    "pbi_per_capita": [8200, 9100, 8800, 7500, 6300],
})

df

---
## 3. Crear DataFrames desde cero

Hay varias formas de crear un DataFrame:

In [ ]:
# Desde un diccionario de listas
df1 = pd.DataFrame({
    "year": [2021, 2022, 2023],
    "inflacion": [50.9, 94.8, 211.4],
    "desempleo": [10.2, 7.0, 6.9],
})
print("Desde diccionario:")
print(df1)

# Desde una lista de diccionarios (cada dict es una fila)
registros = [
    {"programa": "AUH",       "beneficiarios": 4_200_000, "organismo": "ANSES"},
    {"programa": "Progresar", "beneficiarios": 1_100_000, "organismo": "ANSES"},
    {"programa": "Potenciar", "beneficiarios": 1_300_000, "organismo": "MDS"},
]
df2 = pd.DataFrame(registros)
print("\nDesde lista de dicts:")
print(df2)

---
## 4. Leer archivos

En la práctica, los datos vienen de archivos externos. pandas puede leer la gran mayoría de formatos.

In [ ]:
# CSV — el formato más común en portales de datos abiertos
# df = pd.read_csv("datos/indicadores_sociales.csv")

# Opciones útiles:
# df = pd.read_csv("archivo.csv", sep=";")              # separador punto y coma
# df = pd.read_csv("archivo.csv", encoding="latin1")    # encoding de archivos argentinos
# df = pd.read_csv("archivo.csv", decimal=",")          # decimal con coma
# df = pd.read_csv("archivo.csv", skiprows=2)           # saltear primeras filas
# df = pd.read_csv("archivo.csv", nrows=1000)           # leer solo primeras 1000 filas

# Excel
# df = pd.read_excel("archivo.xlsx", sheet_name="Hoja1")

# JSON
# df = pd.read_json("archivo.json")

# Desde una URL directa
# df = pd.read_csv("https://datos.gob.ar/dataset/.../archivo.csv")

print("Ver ejemplos de lectura en el código comentado arriba.")
print("En la práctica, adaptaremos el read_csv según el archivo de cada fuente.")

In [ ]:
# Para esta clase trabajaremos con un dataset simulado de programas sociales
np.random.seed(42)
n = 200

provincias_lista = [
    "Buenos Aires", "Córdoba", "Santa Fe", "Mendoza", "Tucumán",
    "Salta", "Entre Ríos", "Chaco", "Misiones", "Corrientes"
]
programas_lista = ["AUH", "Progresar", "Potenciar Trabajo", "Alimentar", "Crédito Argenta"]

df = pd.DataFrame({
    "id_beneficiario": range(1001, 1001 + n),
    "provincia": np.random.choice(provincias_lista, n),
    "programa": np.random.choice(programas_lista, n),
    "edad": np.random.randint(18, 65, n),
    "genero": np.random.choice(["F", "M", "X"], n, p=[0.58, 0.40, 0.02]),
    "ingreso_mensual": np.random.lognormal(10.2, 0.6, n).round(0),
    "anios_educacion": np.random.randint(6, 18, n),
    "tiene_empleo_formal": np.random.choice([True, False], n, p=[0.35, 0.65]),
    "monto_transferencia": np.random.choice([18000, 25000, 42000, 55000, 10000], n),
    "fecha_alta": pd.date_range("2020-01-01", periods=n, freq="3D"),
})

# Introducir algunos valores faltantes
df.loc[np.random.choice(df.index, 15, replace=False), 'ingreso_mensual'] = np.nan
df.loc[np.random.choice(df.index, 8,  replace=False), 'anios_educacion'] = np.nan

print(f"Dataset creado: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()

---
## 5. Exploración inicial de un dataset

Antes de cualquier análisis, siempre hay que entender la estructura y calidad del dataset.

In [ ]:
# Dimensiones
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print(f"Columnas: {df.columns.tolist()}")

In [ ]:
# Tipos de datos y valores nulos
df.info()

In [ ]:
# Estadísticas descriptivas de variables numéricas
df.describe().round(1)

In [ ]:
# Descripción de variables categóricas
df.describe(include='object')

In [ ]:
# Conteo de valores nulos por columna
nulos = df.isnull().sum()
print("Valores nulos por columna:")
print(nulos[nulos > 0])

In [ ]:
# Frecuencias de variables categóricas
print("Distribución por programa:")
print(df['programa'].value_counts())
print()
print("Distribución por género:")
print(df['genero'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

---
## 6. Selección básica de datos

Hay tres formas principales de seleccionar datos en un DataFrame:
- **`[]`**: seleccionar columnas
- **`.loc[]`**: seleccionar por etiqueta (nombre de columna/índice)
- **`.iloc[]`**: seleccionar por posición numérica

In [ ]:
# Seleccionar una columna → devuelve una Series
provincias = df['provincia']
print(type(provincias))
print(provincias.head())

# Seleccionar múltiples columnas → devuelve un DataFrame
subdf = df[['provincia', 'programa', 'monto_transferencia']]
subdf.head()

In [ ]:
# .loc[filas, columnas] — por etiquetas
df.loc[0:4, ['provincia', 'programa', 'edad']]

In [ ]:
# .iloc[filas, columnas] — por posición
df.iloc[0:5, 1:4]  # primeras 5 filas, columnas 1 a 3

In [ ]:
# Filtrar filas con condiciones
# Beneficiarios del programa AUH
auh = df[df['programa'] == 'AUH']
print(f"Beneficiarios AUH: {len(auh)}")

# Mujeres mayores de 30 años
mujeres_adultas = df[(df['genero'] == 'F') & (df['edad'] > 30)]
print(f"Mujeres mayores de 30: {len(mujeres_adultas)}")

# Provincias del norte
norte = df[df['provincia'].isin(['Tucumán', 'Salta', 'Chaco', 'Misiones', 'Corrientes'])]
print(f"Beneficiarios en el norte: {len(norte)}")

In [ ]:
# Crear nuevas columnas
df['ingreso_total'] = df['ingreso_mensual'].fillna(0) + df['monto_transferencia']
df['edad_grupo'] = pd.cut(df['edad'], bins=[18, 30, 45, 65], labels=['18-30', '31-45', '46-65'])

df[['edad', 'edad_grupo', 'ingreso_mensual', 'monto_transferencia', 'ingreso_total']].head(8)

In [ ]:
# Ordenar por columna
df.sort_values('ingreso_mensual', ascending=False).head(5)[['provincia', 'programa', 'ingreso_mensual']]

---
## 7. Ejercicios

Usá el DataFrame `df` creado en esta clase para responder las siguientes preguntas:

### Ejercicio 1
¿Cuántos beneficiarios hay en total por provincia? Mostrá el resultado ordenado de mayor a menor.

In [ ]:
# Tu solución aquí


### Ejercicio 2
Filtrá el DataFrame para quedarte solo con beneficiarios del programa "Potenciar Trabajo" que no tienen empleo formal. ¿Cuál es su ingreso mensual promedio?

In [ ]:
# Tu solución aquí


### Ejercicio 3
Creá una nueva columna `"alto_monto"` que sea `True` si `monto_transferencia > 40000` y `False` en caso contrario. ¿Qué porcentaje de beneficiarios recibe un monto alto?

In [ ]:
# Tu solución aquí
